In [2]:
import re

with open('the-verdict.txt', 'r', encoding='utf-8') as file:
    raw = file.read()

print(len(raw))
print(raw[0:100])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [3]:
word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
word_array = [item.lower() for item in word_array if item.strip()]
print(word_array[0:20])
print(len(word_array))

['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']
4690


In [4]:
sorted_words = sorted(set(word_array))
sorted_words.extend(["<unk>", "<endoftext>"])
vocab_size = len(sorted_words)
print(vocab_size)

1078


In [5]:
vocab = {token:idx for idx,token in enumerate(sorted_words)}

for idx, (key,value) in enumerate(vocab.items()):
    print(key, value)
    if idx >= 20:
        break

! 0
" 1
' 2
( 3
) 4
, 5
-- 6
. 7
: 8
; 9
? 10
_ 11
a 12
abdication 13
able 14
about 15
above 16
abruptly 17
absolute 18
absorbed 19
absurdity 20


In [6]:
class Tokenizer():
    def __init__(self, vocab_dict):
        # Two dictionaries for changing the format of the data
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab_dict.items()}

    def encode(self, raw):

        # Return list of ids for any passed in raw text
        word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
        word_array = [item.lower() for item in word_array if item.strip()]

        # Handle unkept words
        word_array = [
            item if item in self.str_to_int
            else "<unk>" for item in word_array
        ]

        ids = [self.str_to_int[i] for i in word_array]
        return ids

    def decode(self, ids):
        # Convert list of ids back into text
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"\'])', r'\1', text)
        return text
        

In [7]:
# Lets run a quck demo on an instance of the tokenizer

tokenizer = Tokenizer(vocab)

text1 = "Hello, I am Neel!"
text2 = "How are you today?"
text = " <endoftext> ".join([text1, text2])
print(text)

Hello, I am Neel! <endoftext> How are you today?


In [8]:
tokenizer.encode(text)

[1076, 5, 481, 47, 1076, 0, 1077, 478, 66, 1072, 1076, 10]

In [9]:
tokenizer.decode(tokenizer.encode(text))

'<unk>, i am <unk>! <endoftext> how are you <unk>?'

In [10]:
import tiktoken
bpe_tokenizer = tiktoken.get_encoding('gpt2')

In [11]:
encoded_text = bpe_tokenizer.encode(raw)

In [12]:
# Let's create input-target pairs 

encoded_sample = encoded_text[50:]
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1] # Y is one more than x
print(x)
print(y)

[290, 4920, 2241, 287]
[4920, 2241, 287, 257]


In [13]:
import torch
from torch.utils.data import Dataset, DataLoader

class LLMDataset(Dataset):

    def __init__(self, text, tokenizer, length, stride):

        self.input_ids = []
        self.target_ids = []

        ids = tokenizer.encode(text)

        # Loop through data and append input and target tensors to the id arrays


        for i in range(0, len(ids) - length - 1, stride):

            input = ids[i:i+length]
            target = ids[i+1:i+1+length]

            self.input_ids.append(torch.tensor(input))
            self.target_ids.append(torch.tensor(target))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [14]:
def create_dataloader(text, batch_size=4, length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding('gpt2')

    dataset = LLMDataset(text, tokenizer, length, stride)

    dataloader = DataLoader(
        dataset, batch_size, shuffle, drop_last=drop_last, num_workers=num_workers
    )

    return dataloader

In [15]:
vocab_size = 50257
output_dim = 256

embd = torch.nn.Embedding(vocab_size, output_dim)

context_length = 4
dataloader = create_dataloader(raw, 8, context_length, stride=context_length, shuffle=False)

In [16]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [17]:
print(inputs)
print(inputs.shape)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])


In [18]:
token_embeddings = embd(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [22]:
positional_embd_layer = torch.nn.Embedding(context_length, output_dim)
positional_embd = positional_embd_layer(torch.arange(context_length))

In [23]:
positional_embd.shape

torch.Size([4, 256])

In [ ]:
input_embd = token_embeddings + positional_embd # Uses broadcasting to convert positional embd shape
input_embd.shape

torch.Size([8, 4, 256])